In [1]:
import json
import time
import os

# File paths
Patients_File = "Hospital_patients_data.json"
Bills_File = "Hospital_bills_data.json"
Key_File = "Hospital_key_data.json"
Initial_Key = 1000

class Patient:
    def __init__(self, name, age, gender, contact, address=None, blood_group=None):
        self.name = name
        self.age = age
        self.gender = gender
        self.contact = contact
        self.address = address if address else "N/A"
        self.blood_group = blood_group if blood_group else "N/A"

    def to_dict(self):
        return self.__dict__

    @classmethod
    def from_dict(cls, data):
        return cls(
            name=data["name"],
            age=data["age"],
            gender=data["gender"],
            contact=data["contact"],
            address=data.get("address"),
            blood_group=data.get("blood_group")
        )
class HospitalSystem:

    def __init__(self):
        self.patients = {}  # id: patient object
        self.bills = {}     # bill_id: bill data dict
        self.key = Initial_Key
    def Load_Data(self):
        """Loads patients, bills, and key data from JSON files"""
        try:
            if os.path.exists(Patients_File):
                
                with open(Patients_File, 'r') as f:
                    
                    data = json.load(f)
                    self.patients = {
                        
                        pid: Patient.from_dict(pdata)
                        for pid, pdata in data.items()
                    }

            if os.path.exists(Bills_File):
                with open(Bills_File, 'r') as f:
                    self.bills = json.load(f)

            if os.path.exists(Key_File):
                with open(Key_File, 'r') as f:
                    self.key = int(f.read().strip())
            print(f"Loaded {len(self.patients)} patient records and {len(self.bills)} bills...")
        except Exception as e:
            print(f"Error loading data: {e}")

    def save_data(self):
        """Saves all data to JSON files."""
        try:
            patients_to_save = {
                pid: patient.to_dict()
                for pid, patient in self.patients.items()
            }

            with open(Patients_File, 'w') as f:
                json.dump(patients_to_save, f, indent=4)

            with open(Bills_File, 'w') as f:
                json.dump(self.bills, f, indent=4)

            with open(Key_File, 'w') as f:
                f.write(str(self.key))

            print("All data saved successfully.")
            return True

        except Exception as e:
            print(f"Error saving data: {e}")
            return False

    def generate_patient_id(self):
        """Generates a new patient ID and increments the key."""
        new_id = str(self.key)
        self.key += 1
        return new_id
    def add_patient(self):
        """Allows reception to add a new patient."""
        print("\n\nAdd Patient")

        name = input("Enter patient name: ")

        while True:
            try:
                age = int(input("Enter age: "))
                if age <= 0:
                    print("Invalid input. Age must be positive.")
                    continue
                break
            except ValueError:
                print("Invalid input. Age must be a number.")

        gender = input("Enter gender (M/F/Other): ")
        contact = input("Enter contact number: ")
        address = input("Enter address (optional): ")
        blood_group = input("Enter blood group (optional): ")

        new_id = self.generate_patient_id()
        try:
            new_patient = Patient(name, age, gender, contact, address, blood_group)
            self.patients[new_id] = new_patient
            print(f"\nPatient {name} registered successfully.")
            print(f"Assigned patient ID: {new_id}")

            self.save_data()
        except Exception as e:
            print(f"Error creating patient record: {e}")
            self.key -= 1

    def view_patients(self):
        """Displays a list of all registered patients."""
        print("\n\nView Patients")
        if not self.patients:
            print("No patients registered.")
            return

        header = f"{'ID':<6} | {'Name':<20} | {'Age':<3} | {'Gender':<6} | {'Contact':<12}"
        print(header)
        print("-" * len(header))

        for id_num, patient in self.patients.items():
            print(
                f"{id_num:<6} | {patient.name[:20]:<20} | {patient.age:<3} | {patient.gender:<6} | {patient.contact:<12}"
            )

        print(f"\nTotal Registered Patients: {len(self.patients)}")

    def generate_bill(self):
        """Generates a bill for a patient."""
        print("\n\nGenerate Bill")
        patient_id = input("Enter patient ID for billing: ")

        if patient_id not in self.patients:
            print("Error: patient ID does not exist.")
            return

        while True:

            fee_input = input("Enter services/consultation fee: ")
            fee_clean = fee_input.replace(",", "")
            try:
                fee = float(fee_clean)
                if fee <= 0:
                    print("Fee must be a positive amount.")
                    continue
                break
            except ValueError:
                print("Invalid input. Please enter a numeric fee.")
        bill_id = str(len(self.bills) + 1).zfill(4)  # Auto-generated sequential ID
        bill_date = time.strftime("%Y-%m-%d")
        patient = self.patients[patient_id]

        bill_data = {
            'patient_id': patient_id,
            'amount': fee,
            'date': bill_date,
            'patient_name': patient.name
        }
        self.bills[bill_id] = bill_data
        print("\n--- Bill Summary ---")
        print(f"Bill ID: {bill_id}")
        print(f"Patient: {patient.name}")
        print(f"Amount: ${fee:,.2f}")
        print(f"Date: {bill_date}")
        print("Bill generated successfully.")

        self.save_data()

    def display_menu(self):
        """Displays the main menu and handles user input."""
        while True:
            print("\n\n--- Hospital Management System (Reception) ---")
            print("1. Add Patient")
            print("2. View Patients")
            print("3. Generate Bill")
            print("4. Exit (Save all data before exiting)")

            choice = input("Enter your choice (1-4): ")
            if choice == '1':
                self.add_patient()
            elif choice == '2':
                self.view_patients()
            elif choice == '3':
                self.generate_bill()
            elif choice == '4':
                print("\nSaving data and exiting...")
                self.save_data()
                break
            else:
                print("Invalid choice. Please select 1, 2, 3, or 4.")


# --- Main Execution ---
if __name__ == "__main__":
    app = HospitalSystem()
    app.Load_Data()
    app.display_menu()


Loaded 11 patient records and 5 bills...


--- Hospital Management System (Reception) ---
1. Add Patient
2. View Patients
3. Generate Bill
4. Exit (Save all data before exiting)


Enter your choice (1-4):  1




Add Patient


Enter patient name:  Harsha Sahu
Enter age:  30
Enter gender (M/F/Other):  F
Enter contact number:  92893937100
Enter address (optional):  890, shanti nagar(raj)
Enter blood group (optional):  A+



Patient Harsha Sahu registered successfully.
Assigned patient ID: 1011
All data saved successfully.


--- Hospital Management System (Reception) ---
1. Add Patient
2. View Patients
3. Generate Bill
4. Exit (Save all data before exiting)


...



Generate Bill


Enter patient ID for billing:  1004
Enter services/consultation fee:  500



--- Bill Summary ---
Bill ID: 0010
Patient: 1
Amount: $500.00
Date: 2025-10-29
Bill generated successfully.
All data saved successfully.


--- Hospital Management System (Reception) ---
1. Add Patient
2. View Patients
3. Generate Bill
4. Exit (Save all data before exiting)


Enter your choice (1-4):  4



Saving data and exiting...
All data saved successfully.
